In [5]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.schema import Document
import os

In [6]:
embeddings = GoogleGenerativeAIEmbeddings(
	model="models/gemini-embedding-001",
	google_api_key=os.getenv("GOOGLE_API_KEY")
)

In [3]:
doc1 = Document(
    page_content=(
        "Virat Kohli is one of the most successful and consistent batsmen in IPL history. "
        "Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons."
    ),
    metadata={"team": "Royal Challengers Bangalore"}
)

doc2 = Document(
    page_content=(
        "Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. "
        "He's known for his calm demeanor and ability to play big innings under pressure."
    ),
    metadata={"team": "Mumbai Indians"}
)

doc3 = Document(
    page_content=(
        "MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. "
        "His finishing skills, wicketkeeping, and leadership are legendary."
    ),
    metadata={"team": "Chennai Super Kings"}
)

doc4 = Document(
    page_content=(
        "Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. "
        "Playing for Mumbai Indians, he is known for his yorkers and death-over expertise."
    ),
    metadata={"team": "Mumbai Indians"}
)

doc5 = Document(
    page_content=(
        "Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. "
        "Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player."
    ),
    metadata={"team": "Chennai Super Kings"}
)

In [4]:
documents = [doc1, doc2, doc3, doc4, doc5]

In [8]:
vector_store = Chroma(
    embedding_function=embeddings,
    collection_name="ipl_players",
    persist_directory="./chroma_db"
)

### **Add Documents in Database**

In [9]:
# add documents
vector_store.add_documents(documents)

['c5a403f9-26dc-4d4f-a55c-19d3d6997fa3',
 '439e0ae1-4b7f-444c-ba96-3909b8109972',
 'db1a5dc4-2161-4ac2-b90b-2bde4f1b86fe',
 'f00bdaff-2462-4db1-afa1-3f60b96e5487',
 'f9afdf16-8991-4b81-a817-7a08f2cf8f52']

### **View Documents**

In [10]:
vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['c5a403f9-26dc-4d4f-a55c-19d3d6997fa3',
  '439e0ae1-4b7f-444c-ba96-3909b8109972',
  'db1a5dc4-2161-4ac2-b90b-2bde4f1b86fe',
  'f00bdaff-2462-4db1-afa1-3f60b96e5487',
  'f9afdf16-8991-4b81-a817-7a08f2cf8f52'],
 'embeddings': array([[-0.00982054,  0.02545763,  0.02402782, ...,  0.01414876,
         -0.01560954, -0.00266117],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]], shape=(5, 3072)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the m

### **Search Documents**

In [13]:
vector_store.similarity_search(
    query="Which player is bowler among all-rounders?", 
    k=1
)

[Document(id='f9afdf16-8991-4b81-a817-7a08f2cf8f52', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

In [14]:
# search with score (returns tuples of (Document, score))
# score indicates how similar the document is to the query (lower is more similar)
vector_store.similarity_search_with_score(
    query="Which player is known as Captain Cool?", 
    k=1
)

[(Document(id='db1a5dc4-2161-4ac2-b90b-2bde4f1b86fe', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.4321334958076477)]

In [18]:
vector_store.get(where={"team": "Chennai Super Kings"})

{'ids': ['db1a5dc4-2161-4ac2-b90b-2bde4f1b86fe',
  'f9afdf16-8991-4b81-a817-7a08f2cf8f52'],
 'embeddings': None,
 'documents': ['MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'team': 'Chennai Super Kings'},
  {'team': 'Chennai Super Kings'}]}

### **Update Documents**

In [20]:
updated_doc1 = Document(
    page_content=(
        "Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances."
        "He holds the record for the most runs in IPL history, including multiple centuries in a single season."
        "Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league."
        "His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket."
    ),
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_documents(ids=['c5a403f9-26dc-4d4f-a55c-19d3d6997fa3'], documents=[updated_doc1])

In [21]:
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['c5a403f9-26dc-4d4f-a55c-19d3d6997fa3',
  '439e0ae1-4b7f-444c-ba96-3909b8109972',
  'db1a5dc4-2161-4ac2-b90b-2bde4f1b86fe',
  'f00bdaff-2462-4db1-afa1-3f60b96e5487',
  'f9afdf16-8991-4b81-a817-7a08f2cf8f52'],
 'embeddings': array([[-0.00785507,  0.01899119,  0.02878494, ...,  0.01456527,
         -0.01186557, -0.00414132],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]], shape=(5, 3072)),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances.He holds the record for the most runs in IPL history, including multiple ce

### **Delete Document**

In [22]:
vector_store.delete(ids=['c5a403f9-26dc-4d4f-a55c-19d3d6997fa3'])

In [23]:
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['439e0ae1-4b7f-444c-ba96-3909b8109972',
  'db1a5dc4-2161-4ac2-b90b-2bde4f1b86fe',
  'f00bdaff-2462-4db1-afa1-3f60b96e5487',
  'f9afdf16-8991-4b81-a817-7a08f2cf8f52'],
 'embeddings': array([[-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]], shape=(4, 3072)),
 'documents': ["Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Jasprit Bumrah i

In [ ]:
#